Notebook de limpieza de datos

In [ ]:
import os
import shutil

# Carpeta de origen (datos sin procesar)
carpeta_origen = "../../Datos/Datos oncológicos (sin procesar)"

# Carpeta de destino (datos procesados)
carpeta_destino = "../../Datos/Datos oncológicos (procesados)"

# Crear la carpeta de destino si no existe
os.makedirs(carpeta_destino, exist_ok=True)

# Lista de archivos a copiar (años 2019-2024)
archivos = [
    "GRD_ONCOLOGIA_2019.csv",
    "GRD_ONCOLOGIA_2020.csv",
    "GRD_ONCOLOGIA_2021.csv",
    "GRD_ONCOLOGIA_2022.csv",
    "GRD_ONCOLOGIA_2023.csv",
    "GRD_ONCOLOGIA_2024.csv"
]

# Copiar cada archivo al destino con nuevo nombre
for archivo in archivos:
    origen = os.path.join(carpeta_origen, archivo)
    año = archivo[-8:-4]  # extrae el año del nombre original
    nuevo_nombre = f"GRD_ONCOLOGIA_PROC_{año}.csv"
    destino = os.path.join(carpeta_destino, nuevo_nombre)
    shutil.copy(origen, destino)
    print(f"Archivo {archivo} copiado como {nuevo_nombre} en {carpeta_destino}")

Archivo GRD_ONCOLOGIA_2019.csv copiado como GRD_ONCOLOGIA_PROC_2019.csv en ../../Datos/Datos oncológicos (procesados)
Archivo GRD_ONCOLOGIA_2020.csv copiado como GRD_ONCOLOGIA_PROC_2020.csv en ../../Datos/Datos oncológicos (procesados)
Archivo GRD_ONCOLOGIA_2021.csv copiado como GRD_ONCOLOGIA_PROC_2021.csv en ../../Datos/Datos oncológicos (procesados)
Archivo GRD_ONCOLOGIA_2022.csv copiado como GRD_ONCOLOGIA_PROC_2022.csv en ../../Datos/Datos oncológicos (procesados)
Archivo GRD_ONCOLOGIA_2023.csv copiado como GRD_ONCOLOGIA_PROC_2023.csv en ../../Datos/Datos oncológicos (procesados)
Archivo GRD_ONCOLOGIA_2024.csv copiado como GRD_ONCOLOGIA_PROC_2024.csv en ../../Datos/Datos oncológicos (procesados)


In [ ]:
import pandas as pd
import os
import numpy as np
import re

def tabla_frecuencias(carpeta, archivos, columna):
    """
    Genera una tabla de frecuencias anuales para una columna categórica.
    Filas = categorías, Columnas = años.
    """
    resultados = {}
    for archivo in archivos:
        ruta = os.path.join(carpeta, archivo)
        # Evitar warnings de tipos mezclados
        df = pd.read_csv(ruta, low_memory=False)
        año = archivo[-8:-4]
        # Convertir la columna a string para evitar conflictos de tipos
        resultados[año] = df[columna].astype(str).value_counts()

    # Combinar resultados en un DataFrame
    tabla = pd.DataFrame(resultados).fillna(0).astype(int)
    return tabla

# Ejemplo de uso con MORTALIDAD
carpeta = "../../Datos/Datos oncológicos (procesados)"
archivos = [
    "GRD_ONCOLOGIA_PROC_2019.csv",
    "GRD_ONCOLOGIA_PROC_2020.csv",
    "GRD_ONCOLOGIA_PROC_2021.csv",
    "GRD_ONCOLOGIA_PROC_2022.csv",
    "GRD_ONCOLOGIA_PROC_2023.csv",
    "GRD_ONCOLOGIA_PROC_2024.csv"
]

In [12]:
def evaluar_impacto_nulos(carpeta, archivos, columna, valores_filtro):
    """
    Evalúa cómo se distribuyen las variables objetivo en los registros 
    que coinciden con una lista de valores específicos en una columna dada.
    """
    resultados_objetivos = []

    for archivo in archivos:
        ruta = os.path.join(carpeta, archivo)
        año = archivo[-8:-4]
        df = pd.read_csv(ruta, low_memory=False)

        # 1. Estandarizar y filtrar dinámicamente
        col_data = df[columna].astype(str).str.strip().str.upper()
        df_nulos = df[col_data.isin(valores_filtro)]
        
        total_nulos = len(df_nulos)
        
        if total_nulos == 0:
            continue

        # 2. Contar frecuencias de las variables objetivo
        consumo = df_nulos["CONSUMO_RECURSOS"].value_counts(dropna=False).to_dict()
        severidad = df_nulos["SEVERIDAD"].value_counts(dropna=False).to_dict()
        mortalidad = df_nulos["MORTALIDAD"].value_counts(dropna=False).to_dict()

        # 3. Almacenar resultados
        resultados_objetivos.append({
            "Año": año,
            "Total Faltantes": total_nulos,
            "MORT (0)": mortalidad.get(0, 0),
            "MORT (1)": mortalidad.get(1, 0),
            "SEV (0)": severidad.get(0, 0),
            "SEV (1)": severidad.get(1, 0),
            "SEV (2)": severidad.get(2, 0),
            "SEV (3)": severidad.get(3, 0),
            "CONS (1)": consumo.get(1, 0),
            "CONS (2)": consumo.get(2, 0),
            "CONS (3)": consumo.get(3, 0),
        })

    # 4. Formatear salida
    if resultados_objetivos:
        tabla_obj = pd.DataFrame(resultados_objetivos).fillna(0).astype(int).set_index("Año")
        print(f"\nDISTRIBUCIÓN DE OBJETIVOS EN '{columna}' (Filtro aplicado: {valores_filtro})\n")
        return tabla_obj
    else:
        print(f"No se encontraron registros en '{columna}' que coincidan con el filtro.")
        return None

# **I. Variables objetivo:** Mortalidad, Consumo de recursos, Severidad



## **1. Mortalidad:** 0 vivo, 1 fallecido.
- **Decisión de procesamiento:** Binarizar y eliminar casos nulos (no identificados). En específico la clase positiva (1 - fallecido) se va a componer por los casos de la categoría FALLECIDO, y la clase negativa (0 - vivo) se va a componer por el resto de destinos del paciente (DOMICILIO, HOSPITALIZACION DOMICILIARIA, DERIVACION OTRO HOSPITAL DEL SERVICIO, DERIVACION OTRO HOSPITAL DE LA RED NACIONAL, ALTA VOLUNTARIA, DERIVACION INST. PRIVADA, DERIVACION A OTROS CENTROS, FUGA DEL PACIENTE, DERIVACION INST. PRIVADA (VOLUNTARIO)), exceptuando los casos desconocidos (NO IDENTIFICADA).
- Como se tiene un solo caso en 2020 donde se desconoce el desenlace del paciente (clase NO IDENTIFICADA) se va a eliminar dicho episodio (fila), ya que es una regla importante dentro del aprendizaje automático jamás imputar una variable objetivo.

In [13]:
# --- MODIFICACIÓN DE ARCHIVOS ---
for archivo in archivos:
    ruta = os.path.join(carpeta, archivo)
    df = pd.read_csv(ruta)

    # Eliminar filas con NO IDENTIFICADA
    df = df[df["TIPOALTA"] != "NO IDENTIFICADA"]

    # Crear variable binaria con nombre MORTALIDAD
    df["MORTALIDAD"] = df["TIPOALTA"].apply(lambda x: 1 if x == "FALLECIDO" else 0)

    # Guardar archivo modificado sobrescribiendo el original
    df.to_csv(ruta, index=False)

# --- EJEMPLO DE USO ---
tabla_mortalidad = tabla_frecuencias(carpeta, archivos, "MORTALIDAD")
tabla_mortalidad

C:\Users\Carloto\AppData\Local\Temp\ipykernel_3628\1695831336.py:4: DtypeWarning: Columns (28,29,30,31,32,33,37,39,81,82,83,84,85,86,87,89) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(ruta)
C:\Users\Carloto\AppData\Local\Temp\ipykernel_3628\1695831336.py:4: DtypeWarning: Columns (1,32,33,39,83,84,85,86,87) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(ruta)
C:\Users\Carloto\AppData\Local\Temp\ipykernel_3628\1695831336.py:4: DtypeWarning: Columns (39,83,84,85,86,87) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(ruta)
C:\Users\Carloto\AppData\Local\Temp\ipykernel_3628\1695831336.py:4: DtypeWarning: Columns (1,30,31,32,33,39,85,86,87,89,124,126,127) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(ruta)
C:\Users\Carloto\AppData\Local\Temp\ipykernel_3628\1695831336.py:4: DtypeWarning: Columns (1,32,33,39,4

,2019,2020,2021,2022,2023,2024
MORTALIDAD,,,,,,
0,43448,33275,36817,42534,49059,52074
1,2793,2018,2019,2293,2725,3037


Severidad: 
- **Decisión de procesamiento:** Al igual que en el caso de inmortalidad, los casos DESCONOCIDO serán eliminados, para no imputar datos en las variables objetivo

In [14]:
for archivo in archivos:
    ruta = os.path.join(carpeta, archivo)
    df = pd.read_csv(ruta, low_memory=False)

    # Convertir a string y limpiar espacios
    df["IR_29301_SEVERIDAD"] = df["IR_29301_SEVERIDAD"].astype(str).str.strip()

    # Filtrar solo categorías válidas
    df = df[df["IR_29301_SEVERIDAD"].isin(["0", "1", "2", "3"])]

    # Convertir a entero para trabajar más cómodo
    df["IR_29301_SEVERIDAD"] = df["IR_29301_SEVERIDAD"].astype(int)

    # Renombrar la columna
    df = df.rename(columns={"IR_29301_SEVERIDAD": "SEVERIDAD"})

    # Guardar archivo modificado
    df.to_csv(ruta, index=False)

tabla_frecuencias(carpeta, archivos, "SEVERIDAD")


,2019,2020,2021,2022,2023,2024
SEVERIDAD,,,,,,
0,5740,2773,4544,6297,8240,8805
1,17550,13129,13749,14671,15115,15782
2,14134,11482,12225,13797,16548,17206
3,8817,7909,8318,10061,11881,13316


Consumo de recursos

In [15]:
for archivo in archivos:
    ruta = os.path.join(carpeta, archivo)
    df = pd.read_csv(ruta, low_memory=False)

    # Normalizar formato de IR_29301_PESO
    peso = (
        df["IR_29301_PESO"]
        .astype(str)
        .str.replace(",", ".", regex=False)  # unificar separador decimal
    )
    peso = pd.to_numeric(peso, errors="coerce")

    # Crear categorías por terciles
    try:
        df["CONSUMO_RECURSOS"] = pd.qcut(
            peso,
            q=3,
            labels=[0, 1, 2]  # 0 = bajo, 1 = medio, 2 = alto
        )
    except ValueError:
        # Si no hay suficientes valores distintos para formar terciles
        df["CONSUMO_RECURSOS"] = None

    # Guardar archivo modificado
    df.to_csv(ruta, index=False)

tabla_frecuencias(carpeta, archivos, "CONSUMO_RECURSOS")

,2019,2020,2021,2022,2023,2024
CONSUMO_RECURSOS,,,,,,
0,15498,12068,12994,15388,17332,18515
1,15920,11740,12967,14516,17653,18563
2,14823,11485,12875,14922,16799,18031


### Sexo: Como la cantidad de pacientes con sexo nulo o desconocido es baja y no hay otras variables que den señales del sexo del paciente (ya que por ejemplo hay solo 2 nulos con cáncer de mama pero este no es exclusivamente para mujeres, y 0 casos con cáncer en organos femeninos o masculinos) se descarta esta variable

In [16]:
import os
import pandas as pd

valores_nulos = ["", "NAN", "NONE", "NULL", "DESCONOCIDO"]

# Usamos un diccionario en lugar de una lista para facilitar 
# la creación directa de la tabla transpuesta
resultados = {}

for archivo in archivos:
    ruta = os.path.join(carpeta, archivo)
    año = archivo[-8:-4]

    # Evitar warnings de tipos mezclados
    df = pd.read_csv(ruta, low_memory=False)

    # =========================
    # LIMPIEZA SEXO
    # =========================
    sexo = df["SEXO"]

    mask_null_real = sexo.isna()

    sexo_clean = sexo.copy().astype("object")
    mask_not_null = ~mask_null_real

    sexo_clean.loc[mask_not_null] = (
        sexo_clean.loc[mask_not_null]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    mask_nulos = mask_null_real | sexo_clean.isin(valores_nulos)

    df_nulos = df[mask_nulos].copy()
    total_nulos = len(df_nulos)

    if total_nulos == 0:
        continue

    # =========================
    # CATEGORIA CANCER
    # =========================
    cancer = df_nulos["CATEGORIA_CANCER"].astype(str).str.strip().str.upper()

    mask_hombre = cancer.str.contains("C60-C63", na=False)
    mask_mujer = cancer.str.contains("C51-C58", na=False)
    mask_mama = cancer.str.contains("C50", na=False)

    rec_h = mask_hombre.sum()
    rec_m = mask_mujer.sum()
    rec_total = rec_h + rec_m

    # =========================
    # GUARDAR RESULTADOS
    # =========================
    # Asignamos todas las métricas al año correspondiente
    resultados[año] = {
        "Sexo nulo (Total)": total_nulos,
        "Recup. Hombres": rec_h,
        "Recup. Mujeres": rec_m,
        "Total Recuperables": rec_total,
        "% Recuperado": f"{(rec_total / total_nulos)*100:.2f}%",
        "C50 (Mama)": mask_mama.sum(),
        "C60-C63 (Hombres)": mask_hombre.sum(),
        "C51-C58 (Mujeres)": mask_mujer.sum()

    }

# =========================
# CREAR Y MOSTRAR TABLA
# =========================
# Al pasar el diccionario, los años quedan como columnas automáticamente
tabla = pd.DataFrame(resultados).fillna(0)

print("\nRESUMEN DE REGISTROS RECUPERABLES POR AÑO\n")
# to_markdown() genera un formato limpio, moderno y sin caracteres invasivos
tabla


RESUMEN DE REGISTROS RECUPERABLES POR AÑO



,2019,2020,2022,2023,2024
Sexo nulo (Total),2,3,1,1,17
Recup. Hombres,0,0,0,0,0
Recup. Mujeres,0,0,0,0,0
Total Recuperables,0,0,0,0,0
% Recuperado,0.00%,0.00%,0.00%,0.00%,0.00%
C50 (Mama),0,2,0,0,0
C60-C63 (Hombres),0,0,0,0,0
C51-C58 (Mujeres),0,0,0,0,0


In [17]:
for archivo in archivos:
    ruta = os.path.join(carpeta, archivo)
    df = pd.read_csv(ruta, low_memory=False)
    
    # 1. Filtramos y limpiamos
    sexo_str = df["SEXO"].astype(str).str.strip().str.upper()
    df_limpio = df[df["SEXO"].notna() & ~sexo_str.isin(valores_nulos)].copy()
    
    # 2. Sobrescribimos el archivo original con los datos limpios
    df_limpio.to_csv(ruta, index=False)

# Ahora tu función irá a leer los archivos que ya no tienen nulos
tabla_frecuencias(carpeta, archivos, "SEXO")

,2019,2020,2021,2022,2023,2024
SEXO,,,,,,
MUJER,26037,19707,21740,25398,29342,31058
HOMBRE,20202,15583,17096,19427,22441,24034


Etnia:

Ingeniería de Características: Binarización de la Variable "Etnia"
El análisis exploratorio de la variable "Etnia" evidenció un desbalance de clases extremo, donde la inmensa mayoría de las observaciones corresponde a pacientes sin pertenencia a pueblos originarios. Adicionalmente, se detectó un quiebre estructural en el registro de los años 2022 y 2023, donde el valor por defecto "NINGUNO" fue reemplazado a nivel de sistema por la etiqueta "OTRO", generando una distorsión administrativa temporal.

Dado que la fragmentación en sub-etnias (Mapuche, Rapa Nui, Aymara, etc.) presenta frecuencias individuales estadísticamente insignificantes (menores al 1.5% en su conjunto) para el entrenamiento de algoritmos predictivos, se implementó una estrategia de binarización por agrupación binaria:

Homologación de Clase Mayoritaria: Se unificaron las categorías "NINGUNO" y "OTRO" bajo el valor 0, corrigiendo el error de registro multianual y definiendo la clase de referencia basal (paciente no indígena).

Consolidación de Clase Minoritaria: Todas las denominaciones de pueblos originarios reconocidos por la ley se agruparon bajo el valor 1, creando la nueva característica booleana ES_PUEBLO_ORIGINARIO.

Esta transformación reduce el ruido administrativo, corrige la deriva de datos (Data Drift) de los años 2022-2023 y proporciona al modelo una dimensión sociodemográfica limpia para evaluar posibles correlaciones con barreras de acceso, severidad o consumo de recursos en la atención oncológica.


In [18]:
for archivo in archivos:
    ruta = os.path.join(carpeta, archivo)
    df = pd.read_csv(ruta, low_memory=False)
    
    # 1. Estandarizar
    etnia_str = df["ETNIA"].astype(str).str.strip().str.upper()
    
    # 2. Binarizar: Si es NINGUNO u OTRO -> 0 (No originario). Todo el resto -> 1 (Pueblo originario)
    df["ETNIA"] = np.where(etnia_str.isin(["NINGUNO", "OTRO"]), 0, 1)
    
    # 3. Renombrar la columna para que el modelo (y tú) la entiendan mejor
    df = df.rename(columns={"ETNIA": "ES_PUEBLO_ORIGINARIO"})
    
    # 4. Sobrescribir
    df.to_csv(ruta, index=False)
tabla_frecuencias(carpeta, archivos, "ES_PUEBLO_ORIGINARIO")

,2019,2020,2021,2022,2023,2024
ES_PUEBLO_ORIGINARIO,,,,,,
0,45242,34686,38293,44169,50952,54071
1,997,604,543,656,831,1021


Nacionalidad:

Tras el análisis de la distribución de las variables objetivo (MORTALIDAD, SEVERIDAD, CONSUMO_RECURSOS) dentro de los registros con NACIONALIDAD nula (630 casos en 2019 y 60 en 2020), se procede a la eliminación de estas filas basándose en los siguientes criterios metodológicos:

- Pérdida mínima en la clase minoritaria: En 2019, la clase minoritaria crítica (MORTALIDAD = 1) cuenta con 2.793 casos a nivel global. Los datos faltantes solo contienen 45 de estos casos, lo que representa una pérdida de apenas el 1.6% de la clase. En 2020, la pérdida es aún menor (14 casos, ~0.7%).

- Ausencia de concentración de riesgo: Las métricas de severidad máxima (SEVERIDAD = 3) presentes en los datos a eliminar (96 casos en 2019 y 20 en 2020) representan cerca del 1% del total de casos severos disponibles. No se observa un sesgo sistemático que agrupe los perfiles clínicos de mayor riesgo en la categoría de nacionalidad desconocida.

- Calidad de la representación: Considerando que el volumen total supera los 45.000 registros anuales, remover estas 690 filas purifica el conjunto de datos sin afectar significativamente la varianza. Esto evita introducir ruido mediante imputación en variables demográficas y asegura un conjunto íntegro para la posterior aplicación de técnicas de sobremuestreo (como SMOTE) dentro de los folds de validación cruzada.

Transformación y Binarización de la Variable "Nacionalidad"
- Tras la eliminación de los registros con nacionalidad nula o "DESCONOCIDO" (cuyo impacto en la representatividad de las clases minoritarias objetivo fue evaluado y descartado por ser marginal), se procedió a transformar la variable original. El análisis exploratorio previo reveló un fuerte desbalance de clases, donde más del 96% de las observaciones corresponden a la nacionalidad "Chile", mientras que el resto se fragmenta en múltiples categorías de baja frecuencia (menores al 2% cada una).

- Para evitar la maldición de la dimensionalidad (al aplicar técnicas de One-Hot Encoding sobre categorías con muy baja representatividad) y facilitar la convergencia de los modelos de Machine Learning, se aplicó una estrategia de reducción de cardinalidad mediante binarización. Se generó la nueva característica ES_EXTRANJERO, asignando el valor 0 a la clase mayoritaria basal ("Chile") y el valor 1 a la presencia de cualquier otra nacionalidad. Esta codificación agrupa la varianza de la minoría demográfica, preservando la información del origen del paciente en un formato altamente interpretable para el algoritmo.

In [19]:
nulos = ["DESCONOCIDO"]
evaluar_impacto_nulos(carpeta, archivos, "NACIONALIDAD", nulos)


DISTRIBUCIÓN DE OBJETIVOS EN 'NACIONALIDAD' (Filtro aplicado: ['DESCONOCIDO'])



,Total Faltantes,MORT (0),MORT (1),SEV (0),SEV (1),SEV (2),SEV (3),CONS (1),CONS (2),CONS (3)
Año,,,,,,,,,,
2019,630,585,45,77,176,281,96,265,169,0
2020,60,46,14,9,14,17,20,13,24,0


In [20]:
import numpy as np
valores_nulos = ["", "NAN", "NONE", "NULL", "DESCONOCIDO"]
for archivo in archivos:
    ruta = os.path.join(carpeta, archivo)
    df = pd.read_csv(ruta, low_memory=False)
    # 1. Estandarizar la columna para comparaciones limpias
    nac_str = df["NACIONALIDAD"].astype(str).str.strip().str.upper()
    # 2. Filtrar y eliminar nulos / DESCONOCIDO
    df_limpio = df[df["NACIONALIDAD"].notna() & ~nac_str.isin(valores_nulos)].copy()
    # 3. Binarizar: CHILE = 0, Cualquier otro (Extranjero) = 1
    # Actualizamos nac_str basándonos en el df_limpio
    nac_limpio_str = df_limpio["NACIONALIDAD"].astype(str).str.strip().str.upper()
    df_limpio["NACIONALIDAD"] = np.where(nac_limpio_str == "CHILE", 0, 1)
    # Opcional: Renombrar la columna para mayor claridad en tu modelo
    df_limpio = df_limpio.rename(columns={"NACIONALIDAD": "ES_EXTRANJERO"})
    # 4. Sobrescribir el archivo
    df_limpio.to_csv(ruta, index=False)
print("Columna procesada: Nulos eliminados y convertida a formato binario (0=Chileno, 1=Extranjero).")
tabla_frecuencias(carpeta, archivos, "ES_EXTRANJERO")

Columna procesada: Nulos eliminados y convertida a formato binario (0=Chileno, 1=Extranjero).


,2019,2020,2021,2022,2023,2024
ES_EXTRANJERO,,,,,,
0,44903,34492,37782,43409,49861,52852
1,706,738,1054,1416,1922,2240


PREVISION:
- Durante el análisis exploratorio de la variable "Previsión" (seguro de salud), se identificaron tres desafíos principales: la presencia de registros no identificados, errores de codificación de caracteres (encoding) entre distintos años, y una alta cardinalidad con categorías de baja frecuencia. Para optimizar esta variable de cara al modelamiento predictivo, se implementaron las siguientes transformaciones:
- Eliminación de Nulos y No Identificados: Se procedió a la eliminación de las categorías "DESCONOCIDO", "NO CONSIGNADO" y "NO IDENTIFICADA". En conjunto, estas representan aproximadamente 113 observaciones en el periodo 2019-2024 (menos del 0.05% del total de la muestra). Dada su nula significancia estadística, su remoción no genera pérdida de representatividad en la caracterización de los perfiles de riesgo oncológico.
- Corrección de Encoding y Agrupación Socio-Sanitaria: Se corrigieron los caracteres anómalos derivados de la extracción cruda de los registros GRD (ej. "ELECCIÃN" a "ELECCIÓN"). Posteriormente, se redujo la dimensionalidad agrupando las categorías según el riesgo socioeconómico y el modelo de financiamiento subyacente. Se fusionaron las modalidades Institucional (MAI) y Libre Elección (MLE) de FONASA según su tramo correspondiente (A, B, C y D), consolidando el indicador de vulnerabilidad económica. Adicionalmente, las previsiones de las Fuerzas Armadas (CAPREDENA, DIPRECA, SISA) se agruparon en una única categoría institucional, mientras que "ISAPRE" y "PARTICULAR" se unificaron bajo el perfil de "Financiamiento Privado". Esta reducción de cardinalidad previene la dispersión de los algoritmos basados en árboles y mejora la interpretabilidad de la variable en el análisis de las clases objetivo.

Justificación para la Eliminación de Registros con Previsión "Desconocida"
- La depuración de la variable "Previsión" implicó la eliminación de las categorías "DESCONOCIDO", "NO CONSIGNADO" y "NO IDENTIFICADA", totalizando 113 registros distribuidos entre 2019, 2020 y 2022. La decisión metodológica de remover estas observaciones se sustenta en la evaluación de su impacto directo sobre las clases minoritarias de las variables objetivo:
- Preservación de la señal de Mortalidad: En 2019 (el año con mayor cantidad de faltantes), se pierden únicamente 12 casos con MORTALIDAD = 1 de un total de 2.748 disponibles. En el consolidado de los años afectados, la pérdida total es de apenas 14 eventos fatales, lo que representa un impacto estadístico nulo (aprox. 0.2%) sobre la representatividad de la clase minoritaria.
- Impacto marginal en Severidad: De manera análoga, los casos de riesgo máximo (SEVERIDAD = 3) presentes en la data a eliminar suman apenas 26 registros en total frente a los más de 26.000 casos severos que componen el periodo analizado (una merma cercana al 0.1%).
- Prevención de sesgos por imputación: Al tratarse de un predictor socio-sanitario crítico, imputar valores faltantes mediante métodos estadísticos tradicionales (como la moda o KNN) corre el riesgo de distorsionar artificialmente la correlación entre el nivel de ingresos del paciente y el desenlace clínico. La eliminación directa (Listwise Deletion) garantiza la pureza y veracidad de la variable de cara a la posterior aplicación de algoritmos de Machine Learning.

Las agrupaciones realizadas se justifican metodológicamente de la siguiente manera:
- Tramos FONASA (A, B, C, D): Se fusionaron las modalidades de atención Institucional (MAI) y Libre Elección (MLE) dentro de sus respectivos tramos de ingresos. Desde una perspectiva de riesgo predictivo en salud, el estrato socioeconómico del paciente (representado por la letra del tramo) es un determinante clínico mucho más fuerte que la modalidad de financiamiento específica del evento.
- Fuerzas Armadas (FFAA): Se agruparon las previsiones de CAPREDENA, DIPRECA y SISA bajo la categoría unificada "FFAA". Estas instituciones operan bajo una lógica de financiamiento estatal paralelo y redes hospitalarias institucionales cerradas, compartiendo un comportamiento sistémico homogéneo.
- Sistema Privado (PRIVADO): Se unificaron las categorías "ISAPRE" y "PARTICULAR". Ambas representan a un segmento de la población que accede a la atención de salud mediante primas de seguros privados o gasto directo de bolsillo (Out-of-Pocket), presentando patrones de consumo de recursos y tiempos de acceso radicalmente distintos a los de la red pública tradicional.

In [21]:
nulos_prevision = ["DESCONOCIDO", "NO CONSIGNADO", "NO IDENTIFICADA"]
evaluar_impacto_nulos(carpeta, archivos, "PREVISION", nulos_prevision)


DISTRIBUCIÓN DE OBJETIVOS EN 'PREVISION' (Filtro aplicado: ['DESCONOCIDO', 'NO CONSIGNADO', 'NO IDENTIFICADA'])



,Total Faltantes,MORT (0),MORT (1),SEV (0),SEV (1),SEV (2),SEV (3),CONS (1),CONS (2),CONS (3)
Año,,,,,,,,,,
2019,90,78,12,28,29,16,17,24,24,0
2020,18,17,1,3,3,5,7,5,9,0
2022,5,4,1,0,1,2,2,2,2,0


In [40]:
nulos_prevision = ["NO APLICA", "IGNORADO", "DESCONOCIDO", "NO CONSIGNADO", "NO IDENTIFICADA"]

for archivo in archivos:
    ruta = os.path.join(carpeta, archivo)
    df = pd.read_csv(ruta, low_memory=False)
    
    # A. Estandarizar a texto y limpiar espacios
    prev_str = df["PREVISION"].astype(str).str.strip().str.upper()
    
    # B. Filtrar eliminando los nulos
    df_limpio = df[df["PREVISION"].notna() & ~prev_str.isin(nulos_prevision)].copy()
    
    serie_prev = df_limpio["PREVISION"].astype(str).str.strip().str.upper()
    
    # C. Agrupación (Uniendo FFAA y Privado)
    condiciones = [
        serie_prev.str.contains(r"MAI\) A", na=False, regex=True),
        serie_prev.str.contains(r"MAI\) B|FMLE_B", na=False, regex=True),
        serie_prev.str.contains(r"MAI\) C|FMLE_C", na=False, regex=True),
        serie_prev.str.contains(r"MAI\) D|FMLE_D", na=False, regex=True),
        # CORRECCIÓN: Agregamos "PRIVADO" explícitamente a la regex
        serie_prev.str.contains(r"CAPREDENA|DIPRECA|FFAA|SISA|ISAPRE|PARTICULAR|PRIVADO", na=False, regex=True)
    ]
    
    opciones = [
        "FONASA A",
        "FONASA B",
        "FONASA C",
        "FONASA D",
        "OTRAS PREVISIONES (FFAA/PRIVADO)"
    ]
    
    df_limpio["PREVISION"] = np.select(condiciones, opciones, default=serie_prev)
    
    # D. Sobrescribir
    df_limpio.to_csv(ruta, index=False)

tabla_frecuencias(carpeta, archivos, "PREVISION")

,2019,2020,2021,2022,2023,2024
PREVISION,,,,,,
FONASA A,8085,6215,6151,6792,7226,6938
FONASA B,24750,19404,21637,25640,29886,31955
FONASA C,5140,3956,4725,5098,6297,7059
FONASA D,7137,5320,5990,6643,7722,8338
OTRAS PREVISIONES (FFAA/PRIVADO),374,275,262,305,272,283


SERVICIO SALUD:
Depuración y Reducción de Cardinalidad: Variable "Servicio de Salud"
- El análisis de frecuencia de la variable "Servicio de Salud" reveló inconsistencias de codificación (derivadas de caracteres especiales multianuales) y una alta cardinalidad con 29 categorías únicas. Asimismo, se detectó un volumen de registros inválidos ("DESCONOCIDO", "IGNORADO", "NO APLICA").

- Para optimizar esta característica, se ejecutaron las siguientes acciones metodológicas:

- Validación y Eliminación de Nulos: Se identificaron 69 registros inválidos a lo largo de la cohorte (2019-2024). Para justificar su eliminación (Listwise Deletion), se evaluó su impacto sobre las clases minoritarias objetivo. El análisis demostró una pérdida irrelevante de información: solo 2 casos con desenlace fatal (MORTALIDAD = 1) y 12 casos de riesgo máximo (SEVERIDAD = 3). Al representar menos del 0.03% del total, se procedió a su depuración sin riesgo de introducir sesgos de representación.

- Corrección Vectorizada: Se homologaron los problemas de codificación unificando las denominaciones fragmentadas (ej. CONCEPCIÃN a CONCEPCIÓN).

- Agrupación Geográfica (Macrozonas): Para prevenir la maldición de la dimensionalidad en modelos predictivos y capturar el riesgo clínico asociado a la centralización de los recursos oncológicos, los 29 servicios originales se recodificaron en 5 Macrozonas Epidemiológicas densas: Norte, Centro, Región Metropolitana, Sur y Austral. Esta agrupación concentra la varianza demográfica, modelando de manera eficiente la brecha de acceso territorial.

In [23]:
nulos_prevision = ["NO APLICA", "IGNORADO", "DESCONOCIDO"]
evaluar_impacto_nulos(carpeta, archivos, "SERVICIO_SALUD", nulos_prevision)


DISTRIBUCIÓN DE OBJETIVOS EN 'SERVICIO_SALUD' (Filtro aplicado: ['NO APLICA', 'IGNORADO', 'DESCONOCIDO'])



,Total Faltantes,MORT (0),MORT (1),SEV (0),SEV (1),SEV (2),SEV (3),CONS (1),CONS (2),CONS (3)
Año,,,,,,,,,,
2019,4,4,0,0,3,1,0,2,1,0
2020,15,15,0,0,5,7,3,4,6,0
2021,14,12,2,1,5,5,3,5,3,0
2023,16,16,0,1,5,6,4,9,4,0
2024,20,20,0,1,16,1,2,8,5,0


In [24]:
nulos_servicio = ["DESCONOCIDO", "IGNORADO", "NO APLICA"]

for archivo in archivos:
    ruta = os.path.join(carpeta, archivo)
    df = pd.read_csv(ruta, low_memory=False)
    
    # A. Limpieza básica a mayúsculas
    serv_str = df["SERVICIO_SALUD"].astype(str).str.strip().str.upper()
    
    # B. Filtramos eliminando los inválidos/nulos
    df_limpio = df[df["SERVICIO_SALUD"].notna() & ~serv_str.isin(nulos_servicio)].copy()
    
    # Variable de trabajo
    serie_serv = df_limpio["SERVICIO_SALUD"].astype(str).str.strip().str.upper()
    
    # C. Agrupación por Macrozonas y solución de encoding
    # Buscamos subcadenas seguras para evitar los caracteres raros
    condiciones = [
        # MACROZONA NORTE (Arica a Coquimbo)
        serie_serv.str.contains(r"ARICA|IQUIQUE|ANTOFAGASTA|ATACAMA|COQUIMBO", na=False, regex=True),
        
        # MACROZONA CENTRO (Valpo, O'Higgins, Maule)
        serie_serv.str.contains(r"ACONCAGUA|VALPARAISO|QUILLOTA|HIGGINS|MAULE", na=False, regex=True),
        
        # REGION METROPOLITANA
        serie_serv.str.contains(r"METROPOLITANO", na=False, regex=True),
        
        # MACROZONA SUR (Ñuble a Los Lagos - Osorno/Reloncaví)
        # Usamos "UBLE" para agarrar Ñuble y Ã‘uble. Usamos "CEPCI" para Concepción.
        serie_serv.str.contains(r"UBLE|BIOBIO|CONCEPCI|TALCAHUANO|ARAUCO|ARAUCAN|VALDIVIA|OSORNO|RELONCAV", na=False, regex=True),
        
        # MACROZONA AUSTRAL (Chiloé a Magallanes)
        serie_serv.str.contains(r"CHILO|AYSEN|MAGALLANES", na=False, regex=True)
    ]
    
    opciones = [
        "MACROZONA NORTE",
        "MACROZONA CENTRO",
        "REGION METROPOLITANA",
        "MACROZONA SUR",
        "MACROZONA AUSTRAL"
    ]
    
    # Asignamos el valor. Si alguno quedara fuera (no debería), mantiene su nombre original
    df_limpio["SERVICIO_SALUD"] = np.select(condiciones, opciones, default=serie_serv)
    
    # D. Sobrescribir
    df_limpio.to_csv(ruta, index=False)

tabla_frecuencias(carpeta, archivos, "SERVICIO_SALUD")

,2019,2020,2021,2022,2023,2024
SERVICIO_SALUD,,,,,,
REGION METROPOLITANA,15619,11475,13090,15161,17683,18585
MACROZONA SUR,13010,10831,12300,13774,15525,16861
MACROZONA CENTRO,10309,7414,7448,9202,10880,11663
MACROZONA NORTE,5403,4566,5113,5670,6589,6811
MACROZONA AUSTRAL,1264,929,871,1013,1090,1152


PROVINCIA

PROVINCIA
El análisis de la variable sociodemográfica PROVINCIA expuso una dimensionalidad crítica (54 categorías), lo que representaba un riesgo sustancial de dispersión del vector de entrada (curse of dimensionality) si se aplicaban técnicas convencionales como One-Hot Encoding. Simultáneamente, la variable de destino hospitalario SERVICIO_SALUD fue previamente homologada en 5 Macrozonas Sanitarias.Para optimizar el modelamiento y capturar una señal epidemiológica latente fundamental en la oncología chilena (la centralización de la atención de alta complejidad), se procedió a:Listwise Deletion: Se eliminaron los registros con origen desconocido. El análisis de impacto confirmó que la ausencia máxima de estos datos (30 casos en 2024) representaba un $0.05\%$ de la cohorte, con una afectación estadísticamente insignificante sobre las clases minoritarias de las variables objetivo, descartando la imputación.Homologación Geográfica: Se mapearon las 54 provincias mediante expresiones regulares (RegEx) hacia las mismas 5 Macrozonas utilizadas en SERVICIO_SALUD.Ingeniería de Variable Derivada: Se cruzó la procedencia con el destino de atención para crear la característica binaria TRASLADO_MACROZONA (0 = atención local, 1 = migración a otra macrozona).Prevención de Multicolinealidad: Finalmente, la variable original PROVINCIA fue descartada de la matriz de entrada.Esta derivación dota al algoritmo de una medida robusta sobre el "desarraigo sanitario" y las barreras geográficas de acceso, las cuales se hipotetizan como aceleradores del consumo de recursos y severidad del paciente al ingreso, sin introducir variables correlacionadas en exceso.

In [25]:
nulos_prevision = ["NO APLICA", "IGNORADO", "DESCONOCIDO"]
evaluar_impacto_nulos(carpeta, archivos, "PROVINCIA", nulos_prevision)


DISTRIBUCIÓN DE OBJETIVOS EN 'PROVINCIA' (Filtro aplicado: ['NO APLICA', 'IGNORADO', 'DESCONOCIDO'])



,Total Faltantes,MORT (0),MORT (1),SEV (0),SEV (1),SEV (2),SEV (3),CONS (1),CONS (2),CONS (3)
Año,,,,,,,,,,
2019,13,13,0,3,7,0,3,6,4,0
2020,9,9,0,0,6,2,1,1,2,0
2022,2,2,0,1,0,1,0,0,0,0
2023,10,10,0,0,7,2,1,7,2,0
2024,30,28,2,0,21,4,5,17,5,0


In [26]:
nulos_provincia = ["DESCONOCIDO", "IGNORADO", "NO APLICA"]

for archivo in archivos:
    ruta = os.path.join(carpeta, archivo)
    df = pd.read_csv(ruta, low_memory=False)
    
    # 1. Filtramos eliminando los nulos/desconocidos de PROVINCIA
    prov_str = df["PROVINCIA"].astype(str).str.strip().str.upper()
    df_limpio = df[df["PROVINCIA"].notna() & ~prov_str.isin(nulos_provincia)].copy()
    
    serie_prov = df_limpio["PROVINCIA"].astype(str).str.strip().str.upper()
    
    # 2. Mapeo a Macrozonas de Residencia
    condiciones = [
        # MACROZONA NORTE (Arica a Coquimbo)
        serie_prov.str.contains(r"ARICA|PARINACOTA|IQUIQUE|TAMARUGAL|TOCOPILLA|EL LOA|ANTOFAGASTA|CHANARAL|COPIAPO|HUASCO|ELQUI|LIMARI|CHOAPA", na=False, regex=True),
        
        # MACROZONA CENTRO (Valpo, O'Higgins, Maule)
        serie_prov.str.contains(r"PETORCA|LOS ANDES|SAN FELIPE|QUILLOTA|VALPARAISO|SAN ANTONIO|MARGA MARGA|ISLA DE PASCUA|CACHAPOAL|COLCHAGUA|CARDENAL CARO|CURICO|TALCA|LINARES|CAUQUENES", na=False, regex=True),
        
        # REGION METROPOLITANA
        serie_prov.str.contains(r"SANTIAGO|CHACABUCO|CORDILLERA|MAIPO|MELIPILLA|TALAGANTE", na=False, regex=True),
        
        # MACROZONA SUR (Ñuble a Los Lagos)
        serie_prov.str.contains(r"DIGUILLIN|ITATA|PUNILLA|NUBLE|BIO-BIO|CONCEPCION|ARAUCO|MALLECO|CAUTIN|VALDIVIA|RANCO|OSORNO|LLANQUIHUE|CHILOE|PALENA", na=False, regex=True),
        
        # MACROZONA AUSTRAL (Aysén y Magallanes)
        serie_prov.str.contains(r"COIHAIQUE|AISEN|GENERAL CARRERA|CAPITAN PRAT|ULTIMA ESPERANZA|MAGALLANES|TIERRA DEL FUEGO|ANTARTICA CHILENA", na=False, regex=True)
    ]
    
    opciones = [
        "MACROZONA NORTE",
        "MACROZONA CENTRO",
        "REGION METROPOLITANA",
        "MACROZONA SUR",
        "MACROZONA AUSTRAL"
    ]
    
    # Creamos la columna temporal
    df_limpio["MACROZONA_RESIDENCIA"] = np.select(condiciones, opciones, default="OTRO")
    
    # 3. Derivación del Desarraigo (Data Leakage-safe, todo es conocido al ingreso)
    # Comparamos la macrozona de residencia con la macrozona del hospital (SERVICIO_SALUD)
    # Si son DISTINTAS, le asignamos 1 (Trasladado). Si son iguales, 0.
    df_limpio["TRASLADO_MACROZONA"] = (df_limpio["MACROZONA_RESIDENCIA"] != df_limpio["SERVICIO_SALUD"].str.upper()).astype(int)
    
    # 4. Limpieza de columnas redundantes
    # Borramos PROVINCIA por su alta cardinalidad y MACROZONA_RESIDENCIA por multicolinealidad
    df_limpio = df_limpio.drop(columns=["PROVINCIA", "MACROZONA_RESIDENCIA"])
    
    # 5. Sobrescribir
    df_limpio.to_csv(ruta, index=False)

tabla_frecuencias(carpeta, archivos, "TRASLADO_MACROZONA")

,2019,2020,2021,2022,2023,2024
TRASLADO_MACROZONA,,,,,,
0,44066,33940,37389,43236,50214,53229
1,1526,1266,1433,1582,1543,1813


TIPO_PROCEDENCIA
- La variable "Tipo de Procedencia" (que indica el origen o vía de ingreso del paciente al recinto hospitalario) presentó una alta dimensionalidad funcional (24 categorías), fragmentación por errores de codificación multianual y una proporción marginal de datos inválidos ("DESCONOCIDO", "NO IDENTIFICADO").

El abordaje metodológico de esta variable consistió en dos etapas:

Eliminación de Registros Inválidos (Listwise Deletion): Se identificaron 35 casos nulos concentrados entre 2019 y 2021. La evaluación de impacto confirmó que su remoción no afecta la representatividad de las clases minoritarias, registrando una pérdida de 0 casos en la variable MORTALIDAD = 1 y apenas 3 casos de SEVERIDAD = 3 (riesgo máximo). Dada su irrelevancia estadística (<0.015% de la muestra), los registros fueron descartados.

Reducción de Cardinalidad por Acuidad Clínica (Acuity Level): Para transformar esta característica en un predictor robusto para modelos de Machine Learning, se corrigieron los errores de encoding y se agruparon las 24 vías de ingreso originales en 6 macro-categorías fundamentadas en el nivel de planificación y gravedad basal del ingreso:

Urgencia / Emergencia: Ingresos no planificados (alta probabilidad de descompensación aguda).

Especialidad / Ambulatorio: Derivaciones planificadas desde centros de nivel secundario (CDT, CRS, Consulta Privada).

Traslado de Red: Derivaciones desde otros centros hospitalarios (sugiere requerimiento de mayor complejidad resolutiva).

Atención Primaria: Derivaciones directas desde el nivel básico (CESFAM, Postas).

Programas y Electividad: Ingresos altamente planificados para procedimientos de corta estadía o resolución de listas de espera (CMA, Estrategias CRR).

Institucional y Alternativa: Ingresos desde recintos de aislamiento social (Cárceles, SENAME) o modalidades de hospitalización fuera de cama tradicional (Domiciliaria, Diurna).

Esta reestructuración captura el gradiente de gravedad asociado a la vía de ingreso, mitigando la dispersión del algoritmo en categorías administrativas de baja frecuencia.

In [27]:
nulos = ["NO IDENTIFICADO", "DESCONOCIDO"]
evaluar_impacto_nulos(carpeta, archivos, "TIPO_PROCEDENCIA", nulos)


DISTRIBUCIÓN DE OBJETIVOS EN 'TIPO_PROCEDENCIA' (Filtro aplicado: ['NO IDENTIFICADO', 'DESCONOCIDO'])



,Total Faltantes,MORT (0),MORT (1),SEV (0),SEV (1),SEV (2),SEV (3),CONS (1),CONS (2),CONS (3)
Año,,,,,,,,,,
2019,7,7,0,0,5,2,0,6,0,0
2020,14,14,0,0,9,2,3,5,5,0
2021,14,14,0,4,4,6,0,5,1,0


In [28]:
nulos_procedencia = ["DESCONOCIDO", "NO IDENTIFICADO"]

for archivo in archivos:
    ruta = os.path.join(carpeta, archivo)
    df = pd.read_csv(ruta, low_memory=False)
    
    proc_str = df["TIPO_PROCEDENCIA"].astype(str).str.strip().str.upper()
    df_limpio = df[df["TIPO_PROCEDENCIA"].notna() & ~proc_str.isin(nulos_procedencia)].copy()
    
    serie_proc = df_limpio["TIPO_PROCEDENCIA"].astype(str).str.strip().str.upper()
    
    # C. Agrupación uniendo las categorías minoritarias
    condiciones = [
        # 1. URGENCIA
        serie_proc.str.contains(r"EMERGENCIA|URGENCIA|SAPU|SUR|SUC", na=False, regex=True),
        
        # 2. ESPECIALIDAD
        serie_proc.str.contains(r"ESPECIALIDAD|CONSULTA PRIVADA", na=False, regex=True),
        
        # 3. TRASLADO CLINICO (Agregué "TRASLADO DE RED" explícitamente)
        serie_proc.str.contains(r"OTROS HOSPITALES|OTRAS INSTITUCIONES SALUD|REHABILITAC|TRASLADO DE RED", na=False, regex=True),
        
        # 4. OTRAS PROCEDENCIAS (Agregué los nombres de las categorías exactas a la regex)
        serie_proc.str.contains(r"CESFAM|POSTA|ATENCION PRIMARIA|CMA|AMBULATORIA|CRR|RESOLUCI|ESPERA|PAGO GRD|PROGRAMAS Y ELECTIVA|RCEL|SENAME|HOGARES|DIURNA|DOMICILIARIA|INSTITUCIONAL Y ALTERNATIVA", na=False, regex=True)
    ]
    
    opciones = [
        "URGENCIA",
        "ESPECIALIDAD",
        "TRASLADO DE RED",
        "OTRAS PROCEDENCIAS"
    ]
    
    df_limpio["TIPO_PROCEDENCIA"] = np.select(condiciones, opciones, default=serie_proc)
    
    # D. Sobrescribir
    df_limpio.to_csv(ruta, index=False)

tabla_frecuencias(carpeta, archivos, "TIPO_PROCEDENCIA")

,2019,2020,2021,2022,2023,2024
TIPO_PROCEDENCIA,,,,,,
ESPECIALIDAD,29779,21312,25015,29177,33486,35536
URGENCIA,11941,10325,10322,10853,12705,13621
TRASLADO DE RED,3162,2876,2667,2824,3350,3707
OTRAS PROCEDENCIAS,703,679,804,1964,2216,2178


TIPO DE INGRESO
Depuración y Binarización: Variable "Tipo de Ingreso"

La auditoría de calidad de la variable administrativa TIPO_INGRESO evidenció una alta integridad de los registros, presentando un volumen marginal de valores nulos o desconocidos (máximo de 8 episodios en el año 2024). El análisis de impacto demostró que dicha ausencia no afectaba a la clase positiva de mortalidad y su impacto en severidad máxima era inferior al 0.05%, por lo que se procedió con su eliminación directa (Listwise Deletion).

En cuanto a la distribución de las categorías válidas, se detectó un desbalance crítico en la clase "Obstétrica", la cual representaba apenas el 0.1% de la cohorte anual (entre 29 y 108 casos). Para evitar la creación de reglas estadísticamente débiles y mitigar el riesgo de sobreajuste (overfitting) en los algoritmos de partición, se aplicó una estrategia de binarización basada en la predictibilidad del consumo de recursos de la cama hospitalaria. Las vías de admisión no electivas ("Urgencia" y "Obstétrica") se consolidaron bajo la macro-categoría "NO PROGRAMADO", contrastándola con los ingresos de naturaleza "PROGRAMADA". Esta transformación reduce la cardinalidad y provee al modelo de un predictor binario limpio sobre la estabilización inicial del paciente.

In [29]:
nulos = ["NO IDENTIFICADA", "DESCONOCIDO"]
evaluar_impacto_nulos(carpeta, archivos, "TIPO_INGRESO", nulos)


DISTRIBUCIÓN DE OBJETIVOS EN 'TIPO_INGRESO' (Filtro aplicado: ['NO IDENTIFICADA', 'DESCONOCIDO'])



,Total Faltantes,MORT (0),MORT (1),SEV (0),SEV (1),SEV (2),SEV (3),CONS (1),CONS (2),CONS (3)
Año,,,,,,,,,,
2019,1,1,0,0,0,1,0,0,1,0
2022,3,3,0,0,1,1,1,1,1,0
2023,1,1,0,0,0,0,1,0,1,0
2024,8,8,0,0,2,5,1,1,2,0


In [30]:
nulos_ingreso = ["NO IDENTIFICADA", "DESCONOCIDO"]

for archivo in archivos:
    ruta = os.path.join(carpeta, archivo)
    df = pd.read_csv(ruta, low_memory=False)
    
    # 1. Filtramos eliminando los nulos
    ingreso_str = df["TIPO_INGRESO"].astype(str).str.strip().str.upper()
    df_limpio = df[df["TIPO_INGRESO"].notna() & ~ingreso_str.isin(nulos_ingreso)].copy()
    
    serie_ingreso = df_limpio["TIPO_INGRESO"].astype(str).str.strip().str.upper()
    
    # 2. Binarización de la variable
    condiciones = [
        serie_ingreso.str.contains(r"PROGRAMADA", na=False, regex=True),
        serie_ingreso.str.contains(r"URGENCIA|OBSTETRICA", na=False, regex=True)
    ]
    
    opciones = [
        "PROGRAMADO",
        "NO PROGRAMADO (URGENCIA/OBSTETRICIA)"
    ]
    
    # Asignamos la nueva categoría
    df_limpio["TIPO_INGRESO"] = np.select(condiciones, opciones, default="OTRO")
    
    # 3. Sobrescribir
    df_limpio.to_csv(ruta, index=False)

tabla_frecuencias(carpeta, archivos, "TIPO_INGRESO")

,2019,2020,2021,2022,2023,2024
TIPO_INGRESO,,,,,,
PROGRAMADO,29899,21701,25203,30437,35001,36615
NO PROGRAMADO (URGENCIA/OBSTETRICIA),15685,13491,13605,14378,16755,18419


Especialidad médica
Depuración y Reducción de Cardinalidad: Variable "Especialidad Médica"
El análisis de la variable "Especialidad Médica" (que indica la unidad clínica de egreso del paciente) evidenció una dimensionalidad crítica (más de 70 categorías) y un quiebre estructural en el diccionario de datos: se identificó un cambio de nomenclatura oficial aplicado a partir del año 2020 (ej. transición de "CIRUGIA TORAX" a "CIRUGIA DE TORAX").

Para estandarizar la variable y maximizar su valor predictivo, se ejecutaron las siguientes acciones metodológicas:

Eliminación de Valores Nulos: Se identificaron únicamente 4 registros categorizados como "DESCONOCIDO" (concentrados en 2019). La función de impacto en clases minoritarias demostró una pérdida nula de eventos fatales (MORTALIDAD = 1) y solo 2 casos de severidad máxima. Por consiguiente, se aplicó Listwise Deletion sin riesgo de sesgo.

Ingeniería de Características y Homologación Clínica: Para corregir la fragmentación de nomenclatura multianual y evitar la maldición de la dimensionalidad, las más de 70 especialidades originales se agruparon mediante RegEx (Expresiones Regulares) en 8 Macro-Rutas Clínicas afines al manejo oncológico y perfiles de riesgo:

Oncología y Hematología: Agrupa Oncología Médica, Hematología y Radioterapia (rutas directas de tratamiento sistémico).

Cirugía General y Digestiva: Incluye Cirugía General, Digestiva y Coloproctología (principal vía de resolución para tumores sólidos abdominales).

Cirugía Especializada: Consolida subespecialidades de alta complejidad quirúrgica (Tórax, Neurocirugía, Cabeza y Cuello, Vascular, Maxilofacial).

Ginecología y Obstetricia: Ruta principal para patologías oncológicas prevalentes en la mujer (Cáncer de Mama, Cervicouterino, Ovario).

Urología: Ruta principal para tumores prevalentes masculinos y de tracto urinario (Próstata, Riñón, Vejiga).

Pediatría: Agrupa todas las subespecialidades de atención infantil (excluyendo la Hemato-Oncología Pediátrica, asignada a la ruta oncológica).

Medicina Interna y Crítica: Consolida unidades de soporte vital y manejo médico complejo (Medicina Intensiva, Broncopulmonar, Gastroenterología, Urgencia).

Otras Especialidades: Categoría residual para especialidades de bajo volumen de egreso hospitalario u orientadas a atención ambulatoria (Oftalmología, Dermatología, Odontología).

Esta reducción de cardinalidad consolida la señal estadística de las vías de tratamiento, permitiendo que los modelos basados en árboles capturen el riesgo inherente a la complejidad de cada macro-unidad médica.

In [31]:
nulos = ["DESCONOCIDO"]
evaluar_impacto_nulos(carpeta, archivos, "ESPECIALIDAD_MEDICA", nulos)


DISTRIBUCIÓN DE OBJETIVOS EN 'ESPECIALIDAD_MEDICA' (Filtro aplicado: ['DESCONOCIDO'])



,Total Faltantes,MORT (0),MORT (1),SEV (0),SEV (1),SEV (2),SEV (3),CONS (1),CONS (2),CONS (3)
Año,,,,,,,,,,
2019,5,5,0,0,1,2,2,2,2,0


In [32]:
nulos_especialidad = ["DESCONOCIDO"]

for archivo in archivos:
    ruta = os.path.join(carpeta, archivo)
    df = pd.read_csv(ruta, low_memory=False)
    
    # A. Limpieza a mayúsculas
    esp_str = df["ESPECIALIDAD_MEDICA"].astype(str).str.strip().str.upper()
    
    # B. Filtramos eliminando los nulos (los 4 casos)
    df_limpio = df[df["ESPECIALIDAD_MEDICA"].notna() & ~esp_str.isin(nulos_especialidad)].copy()
    
    # Variable de trabajo
    serie_esp = df_limpio["ESPECIALIDAD_MEDICA"].astype(str).str.strip().str.upper()
    
    # C. Agrupación por Macro-Rutas Clínicas (Regex)
    # EL ORDEN IMPORTA. Las más específicas van primero.
    condiciones = [
        # 1. ONCOLOGIA Y HEMATOLOGIA
        serie_esp.str.contains(r"ONCOLOG|HEMATOLOG|RADIOTERAPIA", na=False, regex=True),
        
        # 2. PEDIATRIA
        serie_esp.str.contains(r"PEDIATR|NEONATOLOG|SENAME", na=False, regex=True),
        
        # 3. CIRUGIA GENERAL Y DIGESTIVA (<- AQUÍ ESTÁ EL ARREGLO)
        # CIRUG.*GENERAL atrapará "CIRUGIA GENERAL", "CIRUGÍA GENERAL", "CIRUGÃA GENERAL", etc.
        serie_esp.str.contains(r"CIRUG.*GENERAL|DIGESTIV|COLOPROCTOLOG|GASTROENTEROLOG", na=False, regex=True),
        
        # 4. GINECOLOGIA Y OBSTETRICIA
        serie_esp.str.contains(r"GINECOLOG|OBSTETRICIA", na=False, regex=True),
        
        # 5. UROLOGIA
        serie_esp.str.contains(r"UROLOG", na=False, regex=True),
        
        # 6. CIRUGIA ESPECIALIZADA 
        serie_esp.str.contains(r"TORAX|CABEZA|CUELLO|MAXILO|NEUROCIRUG|VASCULAR|TRAUMATOLOG|PLASTICA|CARDIO|OTORRINO", na=False, regex=True),
        
        # 7. MEDICINA INTERNA Y CRITICA
        serie_esp.str.contains(r"INTERNA|INTENSIV|RESPIRATORI|BRONCOPULMONAR|CARDIOLOG|NEFROLOG|INFECTOLOG|ENDOCRINOLOG|GERIATRIA|URGENCIA", na=False, regex=True)
    ]
    
    opciones = [
        "ONCOLOGIA Y HEMATOLOGIA",
        "PEDIATRIA",
        "CIRUGIA GENERAL Y DIGESTIVA",
        "GINECOLOGIA Y OBSTETRICIA",
        "UROLOGIA",
        "CIRUGIA ESPECIALIZADA",
        "MEDICINA INTERNA Y CRITICA"
    ]
    
    # Todo lo que no caiga en las 7 anteriores, va a "OTRAS ESPECIALIDADES" (Medicina General, Derma, Oftalmo, etc.)
    df_limpio["ESPECIALIDAD_MEDICA"] = np.select(condiciones, opciones, default="OTRAS ESPECIALIDADES")
    
    # D. Sobrescribir
    df_limpio.to_csv(ruta, index=False)
tabla_frecuencias(carpeta, archivos, "ESPECIALIDAD_MEDICA")

,2019,2020,2021,2022,2023,2024
ESPECIALIDAD_MEDICA,,,,,,
CIRUGIA ESPECIALIZADA,3384,2596,3214,3976,5040,5303
CIRUGIA GENERAL Y DIGESTIVA,16808,13725,15182,17241,19247,19937
GINECOLOGIA Y OBSTETRICIA,5436,4610,4972,5734,6719,7271
MEDICINA INTERNA Y CRITICA,6736,5415,5346,5977,7144,7865
ONCOLOGIA Y HEMATOLOGIA,4283,3050,3371,4028,4346,4892
OTRAS ESPECIALIDADES,3697,1676,2102,2394,2824,3030
PEDIATRIA,369,339,371,381,349,419
UROLOGIA,4866,3781,4250,5084,6087,6317


Tipo de actividad:

Ingeniería de Características: Prevención de Concept Drift en "Tipo de Actividad"
Al analizar la distribución temporal de la variable "Tipo de Actividad", se detectó un cambio estructural en los registros a partir del año 2020: las categorías "Hospitalización Diurna" y "Hospitalización en Urgencia" (presentes con alto volumen en 2019) desaparecen de la cohorte en los años posteriores, siendo absorbidas por las categorías principales debido a actualizaciones en la codificación hospitalaria del MINSAL.

Dado que la metodología del proyecto contempla una división temporal de los datos (Entrenamiento: 2019-2023, Prueba: 2024), mantener "Hospitalización Diurna" como una categoría independiente habría introducido Deriva de Concepto (Concept Drift). El algoritmo habría generado reglas sobre una característica que es inexistente en el conjunto de validación (Ghost Feature), degradando la generalización del modelo.

Para asegurar la consistencia temporal y maximizar la señal predictiva relacionada al consumo de recursos, se procedió a binarizar la variable según el requerimiento de pernoctación hospitalaria:

Hospitalización: Consolidación de ingresos tradicionales y de urgencia (pacientes con uso de cama de dotación).

Atención Ambulatoria (CMA / Diurna): Fusión de la Cirugía Mayor Ambulatoria con la Hospitalización Diurna de 2019, agrupando a todos los pacientes con procedimientos resolutivos transitorios sin pernocte.

Esta reducción garantiza que el espacio de características (Feature Space) sea idéntico y estable entre las fases de entrenamiento y prueba.

In [33]:
for archivo in archivos:
    ruta = os.path.join(carpeta, archivo)
    df = pd.read_csv(ruta, low_memory=False)
    
    # Trabajamos directamente con la serie original pasada a mayúsculas
    serie_act = df["TIPO_ACTIVIDAD"].astype(str).str.strip().str.upper()
    
    # Agrupación por Regímenes de Actividad (Regex) en 2 macro-categorías
    condiciones = [
        # 1. ATENCION AMBULATORIA (Atrapa tanto CMA como DIURNA)
        serie_act.str.contains(r"CMA|AMBULATORIA|DIURNA", na=False, regex=True),
        
        # 2. HOSPITALIZACION (Atrapa "HOSPITALIZACIÓN", "HOSPITALIZACIÃ“N" y "HOSPITALIZACIÃ“N EN URGENCIA")
        serie_act.str.contains(r"HOSPITALIZAC", na=False, regex=True)
    ]
    
    opciones = [
        "ATENCION AMBULATORIA",
        "HOSPITALIZACION"
    ]
    
    # Aplicar el mapeo (lo que no calce, que se quede como está por seguridad)
    df["TIPO_ACTIVIDAD"] = np.select(condiciones, opciones, default=serie_act)
    
    # Sobrescribir
    df.to_csv(ruta, index=False)

tabla_frecuencias(carpeta, archivos, "TIPO_ACTIVIDAD")

,2019,2020,2021,2022,2023,2024
TIPO_ACTIVIDAD,,,,,,
HOSPITALIZACION,39924,32429,34269,38519,43518,46231
ATENCION AMBULATORIA,5655,2763,4539,6296,8238,8803


Servicio ingreso

Depuración y Reducción de Cardinalidad: Variable "Servicio de Ingreso"
La variable "Servicio de Ingreso" (que denota la unidad física o nivel de cuidado donde el paciente fue admitido) presentó una alta fragmentación administrativa con más de 80 subcategorías, sumado a un aumento progresivo de registros nulos ("DESCONOCIDO") hacia el año 2024.

Para asegurar la robustez de la matriz de características, se implementaron las siguientes directrices metodológicas:

Evaluación y Eliminación de Nulos: Se identificó un máximo de 460 valores faltantes en la cohorte de 2024. El análisis de impacto demostró que dicha ausencia concentra únicamente 12 casos con desenlace fatal (MORTALIDAD = 1) y 180 casos de alto riesgo (SEVERIDAD = 3), representando menos del 1.5% de las clases minoritarias. Dada la irrelevancia estadística y el alto riesgo de introducir sesgos de enrutamiento clínico mediante imputación, se procedió con la eliminación directa (Listwise Deletion) de todos los registros desconocidos.

Ingeniería de Características por Complejidad (Acuity Level): Para mitigar la alta cardinalidad y alinear la variable con la lógica de consumo de recursos del sistema GRD, se utilizó la técnica de Expresiones Regulares (RegEx) para consolidar las más de 80 unidades en 5 Macro-Servicios basados en la intensidad del soporte clínico:

Cuidados Críticos (PAC): Agrupa todas las unidades de paciente crítico (UCI, UTI, Coronario, Quemados), que concentran el mayor costo día-cama.

Transitorio y Pabellón: Unidades de corta estadía y recuperación post-anestésica (Hospital de Día, CMA, Recuperación Central).

Urgencia: Unidades de emergencia indiferenciadas, de adultos y pediátricas.

Materno-Infantil: Consolidación de todas las áreas de ginecología, obstetricia, puerperio, pediatría y neonatología.

Hospitalización Médico-Quirúrgica: Fusión de todas las salas de cuidados básicos y medios (Medicina, Cirugía, Oncología, Urología, Pensionado, etc.), representando la cama de dotación tradicional.

Esta reestructuración condensa el peso predictivo de la infraestructura hospitalaria, entregando a los algoritmos una variable ordinal implícita basada en la intensidad del cuidado.

In [34]:
nulos = ["DESCONOCIDO"]
evaluar_impacto_nulos(carpeta, archivos, "SERVICIOINGRESO", nulos)


DISTRIBUCIÓN DE OBJETIVOS EN 'SERVICIOINGRESO' (Filtro aplicado: ['DESCONOCIDO'])



,Total Faltantes,MORT (0),MORT (1),SEV (0),SEV (1),SEV (2),SEV (3),CONS (1),CONS (2),CONS (3)
Año,,,,,,,,,,
2019,4,4,0,1,0,2,1,2,1,0
2020,4,4,0,1,0,1,2,1,1,0
2021,43,40,3,2,12,4,25,6,30,0
2022,337,318,19,6,130,91,110,105,161,0
2023,353,341,12,2,144,81,126,120,168,0
2024,460,448,12,0,154,180,126,137,187,0


In [35]:
nulos_servicio_ingreso = ["DESCONOCIDO"]

for archivo in archivos:
    ruta = os.path.join(carpeta, archivo)
    df = pd.read_csv(ruta, low_memory=False)
    
    # A. Limpieza a mayúsculas
    serv_ing_str = df["SERVICIOINGRESO"].astype(str).str.strip().str.upper()
    
    # B. Filtramos eliminando los nulos
    df_limpio = df[df["SERVICIOINGRESO"].notna() & ~serv_ing_str.isin(nulos_servicio_ingreso)].copy()
    
    # Variable de trabajo
    serie_ing = df_limpio["SERVICIOINGRESO"].astype(str).str.strip().str.upper()
    
    # C. Agrupación por Nivel de Complejidad / Unidad Clínica (Regex)
    condiciones = [
        # 1. CUIDADOS CRITICOS (Atrapa UCI, UTI, Intensivo, Intermedio, Coronario, Quemados)
        serie_ing.str.contains(r"INTENSIV|INTERMEDIO|UTI|UCI|CORONARIO|QUEMADOS", na=False, regex=True),
        
        # 2. TRANSITORIO Y PABELLON (Atrapa Recuperación, CMA, Hospital de Día)
        serie_ing.str.contains(r"RECUPERACION|DIA|CMA", na=False, regex=True),
        
        # 3. URGENCIA
        serie_ing.str.contains(r"EMERGENCIA", na=False, regex=True),
        
        # 4. MATERNO-INFANTIL
        serie_ing.str.contains(r"GINECOLOG|OBSTETRIC|PEDIATR|INFANTIL|INFANCIA|LACTANTE|NEONATOLOG|PUERPERIO|MATERNIDAD|EMBARAZO", na=False, regex=True),
        
        # 5. HOSPITALIZACION MEDICO-QUIRURGICA (Atrapa todo el resto de camas básicas/medias)
        serie_ing.str.contains(r"CIRUG|MEDIC|QUIRURG|ONCOLOG|UROLOG|AGUDOS|PENSIONADO|GERIATRIA|TRAUMATOLOGIA|NEUROCIRUG|TRANSPLANTE|OFTALMOLOG|OTORRINO|NEUROLOG|PSIQUIATRIA|DERIVACION|INFECCIOSO|DIABETES", na=False, regex=True)
    ]
    
    opciones = [
        "CUIDADOS CRITICOS (PAC)",
        "TRANSITORIO Y PABELLON",
        "URGENCIA",
        "MATERNO-INFANTIL",
        "HOSPITALIZACION MEDICO-QUIRURGICA"
    ]
    
    # Si alguna de las 80 categorías no hace match con nada, caerá en "OTRAS UNIDADES" para control de calidad
    df_limpio["SERVICIOINGRESO"] = np.select(condiciones, opciones, default="OTRAS UNIDADES")
    
    # D. Sobrescribir
    df_limpio.to_csv(ruta, index=False)

tabla_frecuencias(carpeta, archivos, "SERVICIOINGRESO")

,2019,2020,2021,2022,2023,2024
SERVICIOINGRESO,,,,,,
CUIDADOS CRITICOS (PAC),1382,1448,1660,2303,3048,3497
HOSPITALIZACION MEDICO-QUIRURGICA,27220,21250,22563,26498,30090,32139
MATERNO-INFANTIL,5651,4803,5435,6098,6789,7103
OTRAS UNIDADES,5487,4215,4617,4258,4999,4629
TRANSITORIO Y PABELLON,3808,2786,4133,4965,6105,6882
URGENCIA,2027,686,357,356,372,324


USOSPABELLON: Se descarta del dataset por su cantidad de nulos

In [36]:
nulos = ["0.0", "11.0"]
evaluar_impacto_nulos(carpeta, archivos, "SERVICIOINGRESO", nulos)

No se encontraron registros en 'SERVICIOINGRESO' que coincidan con el filtro.


In [37]:
tabla_frecuencias(carpeta, archivos, "USOSPABELLON")

,2019,2020,2021,2022,2023,2024
USOSPABELLON,,,,,,
0.0,6,0,3,0,0,0
1.0,25608,3,23205,26810,29989,31084
11.0,1,0,0,0,0,0
2.0,2101,1,2707,3296,4737,4837
200.0,1,0,0,0,0,0
249.0,1,0,0,0,0,0
3.0,1245,2,1194,1361,1886,2212
4.0,8,0,63,85,157,254
5.0,2,0,90,102,118,124


CATEGORIA CANCER

Auditoría y Ajuste de Cardinalidad: 


Variable "Categoría de Cáncer"Previo a la codificación del espacio de características, se auditó la distribución de frecuencias de la variable predictora principal CATEGORIA_CANCER (derivada del código CIE-10 del diagnóstico de ingreso). El análisis evidenció un alto volumen y balance adecuado en la gran mayoría de los sistemas anatómicos (ej. neoplasias digestivas, de mama y hematopoyéticas).No obstante, se detectó una anomalía de extrema baja frecuencia en la categoría C97 (Múltiples localizaciones independientes), la cual agrupaba un promedio de 15 casos anuales ($<0.05\%$ de la cohorte). 

Para mitigar el riesgo de sparse matrices (matrices dispersas) durante el posterior One-Hot Encoding y evitar inestabilidades en los gradientes del modelo, se implementó una técnica de agrupación de colas clínicas. La categoría C97 fue fusionada algorítmicamente con la categoría C76-C80 (Localizaciones mal definidas y secundarias), consolidando un macro-grupo denominado C76-C80 y C97: Tumores Secundarios, Múltiples o Mal Definidos. Las demás categorías se mantuvieron desagregadas dada su relevancia topográfica y volumen representativo para el aprendizaje del algoritmo.

In [39]:
for archivo in archivos:
    ruta = os.path.join(carpeta, archivo)
    df = pd.read_csv(ruta, low_memory=False)
    
    # Textos exactos a buscar y reemplazar
    cat_original_76 = "C76-C80: Localizaciones mal definidas y secundarias"
    cat_original_97 = "C97: Múltiples localizaciones independientes"
    
    nuevo_nombre = "C76-C80 Y C97: TUMORES SECUNDARIOS MULTIPLES O MAL DEFINIDOS"
    
    # Aplicamos el reemplazo
    df["CATEGORIA_CANCER"] = df["CATEGORIA_CANCER"].replace(
        [cat_original_76, cat_original_97], 
        nuevo_nombre
    )
    
    # Sobrescribimos el archivo en la misma carpeta de procesados
    df.to_csv(ruta, index=False)

tabla_frecuencias(carpeta, archivos, "CATEGORIA_CANCER")

,2019,2020,2021,2022,2023,2024
CATEGORIA_CANCER,,,,,,
"C00-C14: Labio, cavidad oral y faringe",640,461,524,641,623,802
C15-C26: Órganos digestivos,12422,9973,10840,12339,14218,15161
C30-C39: Órganos respiratorios e intratorácicos,2250,1689,1854,2057,2566,2586
C40-C41: Hueso y cartílago articular,392,308,310,394,420,486
C43-C44: Melanoma y otras neoplasias malignas de piel,2224,1079,1286,1545,1575,1796
C45-C49: Tejidos mesoteliales y tejidos blandos,490,355,427,467,575,667
C50: Mama,6412,4707,5346,6916,7786,7539
C51-C58: Órganos genitales femeninos,3751,3115,3388,3652,4321,4788
C60-C63: Órganos genitales masculinos,2728,1866,1991,2483,2985,3067
